# Bielik on TPU - Episode 1: RMSNorm on Pallas/TPU
**Before running**: `Runtime` -> `Change runtime type` -> select a `TPU` hardware accelerator.

## 1. Clone the repository

In [ ]:
import os

REPO_URL = "https://github.com/qooba/bielik-on-tpu.git"
REPO_DIR = "/content/bielik-on-tpu"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

## 2. Install the repo as an editable package

In [ ]:
!pip install -e . --no-deps -q

## 3. Install a matched jax/libtpu pair

In [ ]:
!pip install -q -U 'jax[tpu]==0.11.1' -f https://storage.googleapis.com/jax-releases/libtpu_releases.html

import os

os.kill(os.getpid(), 9)

In [ ]:
%cd /content/bielik-on-tpu

import jax

print(jax.devices())

## 4. Import the kernel and read its source

`kernels/normalization/rms_norm.py` holds both implementations side by side: `rms_norm_ref` (plain JAX) and `rms_norm_pallas` (the Pallas/TPU kernel).

In [ ]:
from kernels.normalization.rms_norm import rms_norm_pallas, rms_norm_ref

??rms_norm_ref

In [ ]:
??rms_norm_pallas

## 5. Quick correctness check

`rms_norm_pallas` vs `rms_norm_ref` vs a plain NumPy reimplementation, at Bielik's real `hidden_size`, on this TPU.

In [ ]:
import jax.numpy as jnp
import numpy as np

key = jax.random.PRNGKey(0)
key_x, key_w = jax.random.split(key)
hidden_size = 1536
n_rows = 512
eps = 1e-6

x = jax.random.normal(key_x, (n_rows, hidden_size), dtype=jnp.float32)
w = jax.random.normal(key_w, (hidden_size,), dtype=jnp.float32)

out_pallas = rms_norm_pallas(x, w, eps=eps)
out_ref = rms_norm_ref(x, w, eps=eps)

x_np, w_np = np.asarray(x), np.asarray(w)
variance_np = np.mean(x_np**2, axis=-1, keepdims=True)
out_numpy = x_np / np.sqrt(variance_np + eps) * w_np

print("max|pallas - ref|   =", float(jnp.max(jnp.abs(out_pallas - out_ref))))
print("max|pallas - numpy| =", float(np.max(np.abs(np.asarray(out_pallas) - out_numpy))))

## 6. Run the benchmark

`pallas` vs `jax_native`, real bandwidth numbers on this TPU session - the same sweeps `make benchmark-rms-norm` runs from the command line.

In [ ]:
from benchmarks.normalization.benchmark_rms_norm import run_benchmark

run_benchmark(save_plots=True, plot_dir="docs/plots/normalization/")

## 7. View the plots

In [ ]:
from IPython.display import Image, display

for name in [
    "rmsnorm-bandwidth-vs-hidden-size.png",
    "rmsnorm-bandwidth-vs-rows.png",
    "rms_norm-summary-bielik-config-gb_s.png",
    "rms_norm-summary-bielik-config-tflops.png",
]:
    display(Image(filename=f"docs/plots/normalization/{name}"))